In [1]:
!uv pip install yfinance

Using Python 3.13.3 environment at: /home/michabay/projects/trading_bot/.venv
Audited 1 package in 4ms


In [2]:
import yfinance as yf
import pandas as pd
import datetime as dt
from dateutil.relativedelta import relativedelta
from zoneinfo import ZoneInfo
import os, shutil

In [3]:
def fix_yf_df(yf_df: pd.DataFrame) -> pd.DataFrame:
    # Transform to long format: Date and Ticker as columns, single-level headers
    df = yf_df.stack(future_stack=True, level=0).rename_axis(["Date", "symbols"])
    # Remove the 'Price' name from the columns index
    df.columns.name = None

    if isinstance(df, pd.Series):
        raise TypeError(f"df is of type {type(df)} instead of pd.DataFrame")

    df.sort_values(["symbols", "Date"], inplace=True)
    df.reset_index(inplace=True)
    zi = ZoneInfo("US/Eastern")
    df["Date"] = df["Date"].apply(
        lambda x: dt.datetime.fromisoformat(str(x)).astimezone(zi)
    )

    if "Adj Close" in df.columns:
        del df["Adj Close"]

    df.rename(columns={
        "Open": "open",
        "High": "high",
        "Low": "low",
        "Close": "close",
        "Volume": "volume",
    }, inplace=True)

    return df

In [4]:
symbols = [ "AAPL", "GE", "HPQ", "INTC", "F", "NKE" ]
end_dt = dt.datetime.now(tz=ZoneInfo("America/New_York"))
start_dt = end_dt - relativedelta(years=1, months=11)
freq = "1h"

orig_df = yf.download(
    symbols, group_by="Ticker",
    prepost=True, interval=freq,
    auto_adjust=True,
    start=start_dt, end=end_dt,
)
orig_df

[*********************100%***********************]  6 of 6 completed


Ticker                            NKE                                      \
Price                            Open        High         Low       Close   
Datetime                                                                    
2023-08-29 21:00:00+00:00  102.050000  102.200000  101.770000  102.090000   
2023-08-29 22:00:00+00:00  102.090000  102.160000  102.000000  102.035000   
2023-08-29 23:00:00+00:00  102.020000  102.460000  102.000000  102.160000   
2023-08-30 08:00:00+00:00  101.960000  102.160000  101.590000  101.600000   
2023-08-30 09:00:00+00:00  101.690000  101.760000  101.530000  101.530000   
...                               ...         ...         ...         ...   
2025-07-29 16:30:00+00:00   78.129997   78.525002   77.849998   78.400002   
2025-07-29 17:30:00+00:00   78.400002   78.459999   77.967354   78.074997   
2025-07-29 18:30:00+00:00   78.070000   78.360001   78.055000   78.180000   
2025-07-29 19:30:00+00:00   78.190002   78.480003   78.190002   78.339996   
2025-07-29 20:00:00+00:00   78.350000   78.500000   77.950000   78.050000   

Ticker                                   F                                   \
Price                       Volume    Open   High    Low    Close    Volume   
Datetime                                                                      
2023-08-29 21:00:00+00:00        0  12.020  12.04  12.01  12.0400         0   
2023-08-29 22:00:00+00:00        0  12.040  12.06  12.02  12.0400         0   
2023-08-29 23:00:00+00:00        0  12.050  12.07  12.04  12.0600         0   
2023-08-30 08:00:00+00:00        0  12.100  12.10  12.04  12.0800         0   
2023-08-30 09:00:00+00:00        0  12.070  12.07  12.03  12.0300         0   
...                            ...     ...    ...    ...      ...       ...   
2025-07-29 16:30:00+00:00   810153  11.105  11.14  11.08  11.1150   4889995   
2025-07-29 17:30:00+00:00   874659  11.115  11.14  11.08  11.0899   6601678   
2025-07-29 18:30:00+00:00  2336608  11.090  11.12  11.08  11.0900   5680938   
2025-07-29 19:30:00+00:00  1420512  11.085  11.10  11.06  11.0650  10007854   
2025-07-29 20:00:00+00:00  1260241  11.070  11.10  11.06  11.0900         0   

Ticker                     ...       INTC                                   \
Price                      ...       Open       High        Low      Close   
Datetime                   ...                                               
2023-08-29 21:00:00+00:00  ...  34.300000  34.300000  34.230000  34.260000   
2023-08-29 22:00:00+00:00  ...  34.250000  34.300000  34.230000  34.271500   
2023-08-29 23:00:00+00:00  ...  34.270000  34.300000  34.230000  34.250000   
2023-08-30 08:00:00+00:00  ...  34.300000  34.320000  34.080000  34.110000   
2023-08-30 09:00:00+00:00  ...  34.110000  34.210000  34.110000  34.180000   
...                        ...        ...        ...        ...        ...   
2025-07-29 16:30:00+00:00  ...  20.385000  20.490000  20.370001  20.425800   
2025-07-29 17:30:00+00:00  ...  20.424999  20.459999  20.379999  20.395000   
2025-07-29 18:30:00+00:00  ...  20.395000  20.490000  20.389999  20.434999   
2025-07-29 19:30:00+00:00  ...  20.440001  20.469999  20.379999  20.400000   
2025-07-29 20:00:00+00:00  ...  20.400000  20.770000  20.350000  20.370000   

Ticker                                        GE                          \
Price                         Volume        Open        High         Low   
Datetime                                                                   
2023-08-29 21:00:00+00:00        0.0   92.011170   92.450120   92.011170   
2023-08-29 22:00:00+00:00        0.0   92.043100   92.306465   92.043100   
2023-08-29 23:00:00+00:00        0.0   92.043100   92.314445   92.035120   
2023-08-30 08:00:00+00:00        0.0   91.484436   91.484436   91.300880   
2023-08-30 09:00:00+00:00        0.0   92.067040   92.106940   91.620110   
...                              ...         ...         ...         ...   
2025-07-29 16:30:00+00:00  4513619.0 

In [6]:
out_dir = "yf_data_1hr"
if os.path.exists(out_dir):
    shutil.rmtree(out_dir)

os.mkdir(out_dir)

orig_all_path = f"{out_dir}/ORIG_ALL_SYMBOLS.csv"
orig_df.to_csv(orig_all_path, index=False)
print(f"> Exported {orig_all_path}")

> Exported yf_data_1hr/ORIG_ALL_SYMBOLS.csv


In [7]:
fixed_df = fix_yf_df(orig_df)

for symb in symbols:
    sliced_df = fixed_df[fixed_df["symbols"] == symb]
    del sliced_df["symbols"]
    csv_path = f"{out_dir}/fixed_{symb}.csv"
    sliced_df.to_csv(csv_path, index=False)
    print(f"> Exported {csv_path}")

all_path = f"{out_dir}/ALL_SYMBOLS.csv"
fixed_df.to_csv(all_path, index=False)
print(f"> Exported {all_path}")

> Exported yf_data_1hr/fixed_AAPL.csv
> Exported yf_data_1hr/fixed_GE.csv
> Exported yf_data_1hr/fixed_HPQ.csv
> Exported yf_data_1hr/fixed_INTC.csv
> Exported yf_data_1hr/fixed_F.csv
> Exported yf_data_1hr/fixed_NKE.csv
> Exported yf_data_1hr/ALL_SYMBOLS.csv
